<a href="https://colab.research.google.com/github/FabrizzioBurgosUni/IA/blob/main/Healthcare_Inteligencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================================
# CELDA 1 - INSTALAR LIBRERÍAS
# =========================================
!pip install -q transformers accelerate sentence-transformers faiss-cpu pandas torch mistralai

In [2]:
# =========================================
# CELDA 2 - SUBIR DATASET
# =========================================
from google.colab import files
uploaded = files.upload()  # Sube healthcare_dataset.csv

Saving healthcare_dataset.csv to healthcare_dataset (2).csv


In [3]:
# =========================================
# CELDA 3 - LEER Y NORMALIZAR DATASET (3K filas)
# =========================================
import pandas as pd
import numpy as np

df = pd.read_csv('healthcare_dataset.csv', nrows=3000)  # ← solo esto cambia
print(f"Dataset cargado: {df.shape}")

Dataset cargado: (3000, 15)


In [4]:
# =========================================
# CELDA 4 - CONVERTIR A DOCUMENTOS AGRUPADOS
# =========================================
import pandas as pd
import numpy as np

# --- Recalcular todo lo necesario desde cero ---
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])
df['Discharge Date']    = pd.to_datetime(df['Discharge Date'])
df['Days of Stay']      = (df['Discharge Date'] - df['Date of Admission']).dt.days
df['Billing Amount']    = df['Billing Amount'].abs().round(2)
df['Admission Year']    = df['Date of Admission'].dt.year

print("Columnas:", df.columns.tolist())
print(f"Filas: {len(df)}")

documents = []

# --- Grupo 1: Por condición médica ---
g1 = df.groupby('Medical Condition').agg(
    Total_Pacientes=('Name', 'count'),
    Edad_Promedio=('Age', 'mean'),
    Facturacion_Promedio=('Billing Amount', 'mean'),
    Dias_Estadia_Promedio=('Days of Stay', 'mean'),
    Medicamento_Mas_Comun=('Medication', lambda x: x.mode()[0])
).reset_index()

for _, row in g1.iterrows():
    documents.append(
        f"CONDICION_MEDICA: {row['Medical Condition']} | "
        f"TOTAL_PACIENTES: {int(row['Total_Pacientes'])} | "
        f"EDAD_PROMEDIO: {row['Edad_Promedio']:.1f} años | "
        f"FACTURACION_PROMEDIO: ${row['Facturacion_Promedio']:.2f} | "
        f"DIAS_ESTADIA_PROMEDIO: {row['Dias_Estadia_Promedio']:.1f} días | "
        f"MEDICAMENTO_MAS_USADO: {row['Medicamento_Mas_Comun']}"
    )

# --- Grupo 2: Por tipo de admisión ---
g2 = df.groupby('Admission Type').agg(
    Total_Casos=('Name', 'count'),
    Facturacion_Promedio=('Billing Amount', 'mean'),
    Dias_Estadia_Promedio=('Days of Stay', 'mean')
).reset_index()

for _, row in g2.iterrows():
    documents.append(
        f"TIPO_ADMISION: {row['Admission Type']} | "
        f"TOTAL_CASOS: {int(row['Total_Casos'])} | "
        f"FACTURACION_PROMEDIO: ${row['Facturacion_Promedio']:.2f} | "
        f"DIAS_ESTADIA_PROMEDIO: {row['Dias_Estadia_Promedio']:.1f} días"
    )

# --- Grupo 3: Por seguro médico ---
g3 = df.groupby('Insurance Provider').agg(
    Total_Pacientes=('Name', 'count'),
    Facturacion_Total=('Billing Amount', 'sum'),
    Facturacion_Promedio=('Billing Amount', 'mean')
).reset_index()

for _, row in g3.iterrows():
    documents.append(
        f"SEGURO_MEDICO: {row['Insurance Provider']} | "
        f"TOTAL_PACIENTES: {int(row['Total_Pacientes'])} | "
        f"FACTURACION_TOTAL: ${row['Facturacion_Total']:.2f} | "
        f"FACTURACION_PROMEDIO: ${row['Facturacion_Promedio']:.2f}"
    )

# --- Grupo 4: Por condición + tipo de admisión ---
g4 = df.groupby(['Medical Condition', 'Admission Type']).agg(
    Total=('Name', 'count'),
    Facturacion_Promedio=('Billing Amount', 'mean'),
    Resultado_Normal=('Test Results', lambda x: (x == 'Normal').sum()),
    Resultado_Anormal=('Test Results', lambda x: (x == 'Abnormal').sum())
).reset_index()

for _, row in g4.iterrows():
    documents.append(
        f"CONDICION_MEDICA: {row['Medical Condition']} | "
        f"TIPO_ADMISION: {row['Admission Type']} | "
        f"TOTAL_CASOS: {int(row['Total'])} | "
        f"FACTURACION_PROMEDIO: ${row['Facturacion_Promedio']:.2f} | "
        f"RESULTADOS_NORMALES: {int(row['Resultado_Normal'])} | "
        f"RESULTADOS_ANORMALES: {int(row['Resultado_Anormal'])}"
    )

# --- Grupo 5: Por medicamento ---
g5 = df.groupby('Medication').agg(
    Total_Pacientes=('Name', 'count'),
    Condicion_Mas_Comun=('Medical Condition', lambda x: x.mode()[0]),
    Facturacion_Promedio=('Billing Amount', 'mean')
).reset_index()

for _, row in g5.iterrows():
    documents.append(
        f"MEDICAMENTO: {row['Medication']} | "
        f"TOTAL_PACIENTES: {int(row['Total_Pacientes'])} | "
        f"CONDICION_MAS_COMUN: {row['Condicion_Mas_Comun']} | "
        f"FACTURACION_PROMEDIO: ${row['Facturacion_Promedio']:.2f}"
    )

# --- Grupo 6: Por año ---
g6 = df.groupby('Admission Year').agg(
    Total_Admisiones=('Name', 'count'),
    Facturacion_Total=('Billing Amount', 'sum'),
    Facturacion_Promedio=('Billing Amount', 'mean'),
    Dias_Promedio=('Days of Stay', 'mean')
).reset_index()

for _, row in g6.iterrows():
    documents.append(
        f"AÑO: {int(row['Admission Year'])} | "
        f"TOTAL_ADMISIONES: {int(row['Total_Admisiones'])} | "
        f"FACTURACION_TOTAL: ${row['Facturacion_Total']:.2f} | "
        f"FACTURACION_PROMEDIO: ${row['Facturacion_Promedio']:.2f} | "
        f"DIAS_ESTADIA_PROMEDIO: {row['Dias_Promedio']:.1f} días"
    )

print(f"\n✅ Total documentos para RAG: {len(documents)}")
print("\nEjemplo:", documents[0])

Columnas: ['Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results', 'Days of Stay', 'Admission Year']
Filas: 3000

✅ Total documentos para RAG: 43

Ejemplo: CONDICION_MEDICA: Arthritis | TOTAL_PACIENTES: 495 | EDAD_PROMEDIO: 53.3 años | FACTURACION_PROMEDIO: $25429.28 | DIAS_ESTADIA_PROMEDIO: 15.0 días | MEDICAMENTO_MAS_USADO: Penicillin


In [5]:
# =========================================
# CELDA 5 - EMBEDDINGS
# =========================================
import os
import numpy as np
from sentence_transformers import SentenceTransformer

EMBEDDINGS_PATH = "embeddings_healthcare.npy"
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

if os.path.exists(EMBEDDINGS_PATH):
    embeddings = np.load(EMBEDDINGS_PATH)
    print(f"✅ Embeddings cargados desde disco: {embeddings.shape}")
else:
    embeddings = embedding_model.encode(
        documents,
        batch_size=64,
        show_progress_bar=True
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print(f"✅ Embeddings generados y guardados: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embeddings cargados desde disco: (43, 384)


In [6]:
# =========================================
# CELDA 6 - ÍNDICE FAISS
# =========================================
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # Pocos docs → búsqueda exacta está bien
index.add(np.array(embeddings, dtype=np.float32))

print(f"✅ FAISS listo con {index.ntotal} vectores")

✅ FAISS listo con 43 vectores


In [10]:
# =========================================
# CELDA 7 - MISTRAL VÍA REQUESTS (sin librería)
# =========================================
import requests

MISTRAL_API_KEY = "HF_TOKEN"

def llamar_mistral(system_prompt, user_prompt):
    response = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {MISTRAL_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "mistral-small-latest",
            "temperature": 0.2,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ]
        }
    )
    return response.json()["choices"][0]["message"]["content"]

print("✅ Mistral via requests listo")

✅ Mistral via requests listo


In [11]:
# =========================================
# CELDA 8 - FUNCIÓN RAG
# =========================================
import numpy as np

def ask_rag(question, k=5):

    # 1. Embedding de la pregunta
    q_emb = embedding_model.encode([question])

    # 2. Buscar documentos similares
    D, I = index.search(np.array(q_emb, dtype=np.float32), k=k)
    context = "\n".join([documents[i] for i in I[0]])

    # 3. Llamar Mistral
    respuesta = llamar_mistral(
        system_prompt="""Eres un asistente experto en análisis de datos de salud.
Tienes acceso a estadísticas hospitalarias: condiciones médicas, admisiones, medicamentos, facturación y resultados.
Responde de forma clara y con números cuando el contexto los tenga.
Si no tienes la información, dilo honestamente.""",
        user_prompt=f"DATOS RELEVANTES:\n{context}\n\nPREGUNTA: {question}"
    )

    return respuesta

In [12]:
# =========================================
# CELDA 9 - PROBAR
# =========================================
print(ask_rag("¿Cuál es la condición médica más frecuente?"))

La condición médica más frecuente es **Obesidad** con un total de **490 casos** (134 Elective + 170 Emergency + 186 Urgent).

*Desglose:*
- Obesidad: 490 casos
- Cáncer: 171 casos
- Diabetes: 158 casos
